In [2]:
import numpy as np
import matplotlib.pyplot as plt
import os
import h5py
import fipcore as fpc
import pyspedas
from pyspedas.projects import mms

In [3]:
trange = ['2015-12-09/05:03:55', '2015-12-09/05:03:59']

In [3]:
fpi_vars = mms.fpi(trange=trange, 
                            datatype=['des-dist', 'dis-dist', 'des-moms', 'dis-moms'], 
                            level='l2', 
                            data_rate='brst',
                            varnames=['mms1_des_numberdensity_brst',
                                       'mms1_dis_bulkv_gse_brst',
                                       'mms1_des_bulkv_gse_brst'],
                            time_clip=True)

02-Jun-26 13:07:26: Loading files for group: probe: 1, drate: brst, level: l2, datatype: des-dist, after sorting and filtering:
02-Jun-26 13:07:26: /Users/rejohn/Data_Speedas/mms/mms1/fpi/brst/l2/des-dist/2015/12/09/mms1_fpi_brst_l2_des-dist_20151209050044_v3.3.0.cdf
02-Jun-26 13:07:26: Loading files for group: probe: 1, drate: brst, level: l2, datatype: dis-dist, after sorting and filtering:
02-Jun-26 13:07:26: /Users/rejohn/Data_Speedas/mms/mms1/fpi/brst/l2/dis-dist/2015/12/09/mms1_fpi_brst_l2_dis-dist_20151209050044_v3.3.0.cdf
02-Jun-26 13:07:26: Loading files for group: probe: 1, drate: brst, level: l2, datatype: des-moms, after sorting and filtering:
02-Jun-26 13:07:26: /Users/rejohn/Data_Speedas/mms/mms1/fpi/brst/l2/des-moms/2015/12/09/mms1_fpi_brst_l2_des-moms_20151209050044_v3.3.0.cdf
02-Jun-26 13:07:26: Loading files for group: probe: 1, drate: brst, level: l2, datatype: dis-moms, after sorting and filtering:
02-Jun-26 13:07:26: /Users/rejohn/Data_Speedas/mms/mms1/fpi/brst/l2/

In [4]:
pyspedas.tplot_names()

0 : mms1_des_numberdensity_brst
1 : mms1_des_bulkv_gse_brst
2 : mms1_dis_bulkv_gse_brst


['mms1_des_numberdensity_brst',
 'mms1_des_bulkv_gse_brst',
 'mms1_dis_bulkv_gse_brst']

In [5]:
edp_vars = mms.edp(trange=trange,
                            probe=1,
                            data_rate='brst',
                            varnames=['mms1_edp_dce_gse_brst_l2', 'mms1_edp_dce_par_epar_brst_l2'],
                            time_clip=True)

02-Jun-26 13:07:40: Loading files for group: probe: 1, drate: brst, level: l2, datatype: dce, after sorting and filtering:
02-Jun-26 13:07:40: /Users/rejohn/Data_Speedas/mms/mms1/edp/brst/l2/dce/2015/12/09/mms1_edp_brst_l2_dce_20151209050044_v2.2.0.cdf


In [6]:
fgm_vars = mms.fgm(trange=trange, probe=1, data_rate='brst',
                            varformat='mms1_fgm_b_gse_brst_l2', time_clip=True)

02-Jun-26 13:07:57: Loading files for group: probe: 1, drate: brst, level: l2, datatype: , after sorting and filtering:
02-Jun-26 13:07:57: /Users/rejohn/Data_Speedas/mms/mms1/fgm/brst/l2/2015/12/09/mms1_fgm_brst_l2_20151209050044_v4.22.0.cdf


In [13]:
bin_width_frac = 0.25
species = 'e'
project_root = os.path.dirname(os.getcwd())
data_dir = os.path.join(project_root, "data")
hfile_pref = f'mms_fpc_{species}_{bin_width_frac:.2f}'
hfile = os.path.join(data_dir, fpc.mms_name_make(hfile_pref, trange[0], trange[1]))
with h5py.File(hfile, "r") as f:
    print(list(f["meta"].keys()))

['bin_width_frac', 'data_rate', 'fac2lmn', 'interleave_info', 'level', 'lmn2fac', 'probe', 'species', 'time', 'trange', 'vth_lim', 'vth_mean', 'vthe', 'vvol']


In [22]:
fgm_vars = mms.fgm(trange=trange, probe=1, data_rate='brst',
                            varformat='mms1_fgm_b_gse_brst_l2', time_clip=True)       

02-Jun-26 18:24:58: Loading files for group: probe: 1, drate: brst, level: l2, datatype: , after sorting and filtering:
02-Jun-26 18:24:58: /Users/rejohn/Data_Speedas/mms/mms1/fgm/brst/l2/2015/12/09/mms1_fgm_brst_l2_20151209050044_v4.22.0.cdf


In [23]:
fgm_vars

['mms1_fgm_b_gse_brst_l2',
 'mms1_fgm_b_gse_brst_l2_bvec',
 'mms1_fgm_b_gse_brst_l2_btot']

In [24]:
import numpy as np
from pyspedas.projects import mms
from pyspedas import tplot_rename, fac_matrix_make, get_data

from fipcore.analysis.jvec_proc import compute_jvec, jvec_to_fac, jvec_to_lmn
from fipcore.utils.io_utils import h5sav
from fipcore.utils.coord_utils import lmn_matrix_make
from fipcore.utils.helper_utils import downsample_cad

In [36]:

def mms_jvec_main(trange, probe='1', data_rate='brst', level='l2'):

    # Computing current density vector in GSE coordinates
    jvec_gse = compute_jvec(trange, probe=probe, data_rate=data_rate, level=level)

    # Read in magnetic field data
    fgm_vars = mms.fgm(trange=trange, probe=probe, data_rate=data_rate, level=level,
        varnames='mms1_fgm_b_gse_brst_l2', time_clip=True, 
        get_support_data=True, no_update=True)
    tplot_rename('mms1_fgm_b_gse_brst_l2_bvec', 'bvec_gse')
    # downsample_cad('bvec_gse', 'bulk_ve_gse', trange, newname='bvec_gse_dwn')

    # Rotate to FAC coordinates
    fac_mat_name = 'fac_mat_var'
    fac_matrix_make(mag_var_name='bvec_gse_dwn', other_dim='Xgse', 
                             newname=fac_mat_name) # Creating FAC matrix
    _, fac_mat = get_data(fac_mat_name)
    print(type(fac_mat))
    print(fac_mat.shape)
    # jvec_fac = jvec_to_fac(jvec_gse, fac_mat_name)

    # Rotate to LMN coordinates
    # lmn_mat_name = 'lmn_mat_var'
    # lmn_matrix_make('bvec_gse_dwn', trange, newname=lmn_mat_name)
    # _, jvec_lmn = jvec_to_lmn(jvec_gse, lmn_mat_name)

    # # Save to .h5 file
    # dat_grps = {
    #     "gse": {
    #         "jvec_gse": jvec_gse
    #     },
    #     "fac": {
    #         "jvec_fac": jvec_fac
    #     },
    #     "lmn": {
    #         "jvec_lmn": jvec_lmn
    #     }
    # }
    # h5sav(hfile, dat_grps)

    # return hfile

In [46]:
probe = '1'
data_rate = 'brst'
level = 'l2'
trange = ['2015-12-09/05:03:55', '2015-12-09/05:03:59']

# Read in FPI data
fpi_vars = mms.fpi(trange= trange, probe=probe, data_rate=data_rate, level=level,
datatype=['des-moms'], time_clip=True, varnames='mms1_des_bulkv_gse_brst', 
get_support_data=True, no_update=True)
tplot_rename('mms1_des_bulkv_gse_brst', 'bulk_ve_gse')

fgm_vars = mms.fgm(trange=trange, probe=probe, data_rate=data_rate, level=level,
        varnames='mms1_fgm_b_gse_brst_l2', time_clip=True, 
        get_support_data=True, no_update=True)
tplot_rename('mms1_fgm_b_gse_brst_l2_bvec', 'bvec_gse')
downsample_cad('bvec_gse', 'bulk_ve_gse', trange, newname='bvec_gse_dwn')

# Rotate to FAC coordinates
fac_mat_name = 'fac_mat_var'
fac_matrix_make(mag_var_name='bvec_gse_dwn', other_dim='Xgse', 
                        newname=fac_mat_name) # Creating FAC matrix
_, fac_mat = get_data(fac_mat_name)
print(type(fac_mat))
print(fac_mat.shape)

03-Jun-26 10:15:25: Searching for local files...
03-Jun-26 10:15:25: Loading: /Users/rejohn/Data_Speedas/mms/mms1/fpi/brst/l2/des-moms/2015/12/09/mms1_fpi_brst_l2_des-moms_20151209050044_v3.3.0.cdf
03-Jun-26 10:15:25: Loading files for group: probe: 1, drate: brst, level: l2, datatype: des-moms, after sorting and filtering:
03-Jun-26 10:15:25: /Users/rejohn/Data_Speedas/mms/mms1/fpi/brst/l2/des-moms/2015/12/09/mms1_fpi_brst_l2_des-moms_20151209050044_v3.3.0.cdf
03-Jun-26 10:15:25: The name mms1_des_errorflags_brst is currently not in pyspedas
03-Jun-26 10:15:25: The name mms1_des_compressionloss_brst is currently not in pyspedas
03-Jun-26 10:15:25: The name mms1_des_pitchangdist_lowen_brst is currently not in pyspedas
03-Jun-26 10:15:25: The name mms1_des_pitchangdist_miden_brst is currently not in pyspedas
03-Jun-26 10:15:25: The name mms1_des_pitchangdist_highen_brst is currently not in pyspedas
03-Jun-26 10:15:25: The name mms1_des_errorflags_brst_moms is currently not in pyspedas
0

<class 'numpy.ndarray'>
(133, 3, 3)


In [34]:
fgm_vars

['mms1_fgm_b_gse_brst_l2',
 'mms1_fgm_b_gse_brst_l2_bvec',
 'mms1_fgm_b_gse_brst_l2_btot']